# Feature Engineering - VariantClassifier

Este notebook demonstra o processo de feature engineering para variantes genômicas.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import sys
sys.path.append('../src')

from modeling.preprocessing import VariantPreprocessor

print('Bibliotecas importadas com sucesso!')

## 1. Carregamento dos Dados

In [ ]:
# Carregar dados brutos
train_df = pd.read_csv('../data/splits/train.csv')
val_df = pd.read_csv('../data/splits/val.csv')

print(f'Train shape: {train_df.shape}')
print(f'Validation shape: {val_df.shape}')

train_df.head()

## 2. Inicialização do Preprocessor

In [ ]:
# Inicializar preprocessor
preprocessor = VariantPreprocessor()

print('Preprocessor inicializado com sucesso!')
print(f'\nFeatures numéricas: {len(preprocessor.numerical_features)}')
print(f'Features categóricas: {len(preprocessor.categorical_features)}')
print(f'Features booleanas: {len(preprocessor.boolean_features)}')

## 3. Fit e Transform

In [ ]:
# Separar features e target
X_train = train_df.drop('pathogenicity', axis=1)
y_train = train_df['pathogenicity']

X_val = val_df.drop('pathogenicity', axis=1)
y_val = val_df['pathogenicity']

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')

In [ ]:
# Fit preprocessor nos dados de treino
X_train_processed = preprocessor.fit_transform(X_train)

print('Preprocessor fitado com sucesso!')
print(f'X_train_processed shape: {X_train_processed.shape}')
print(f'\nDtypes após processamento:')
print(X_train_processed.dtypes.value_counts())

In [ ]:
# Transformar dados de validação
X_val_processed = preprocessor.transform(X_val)

print(f'X_val_processed shape: {X_val_processed.shape}')
print(f'\nDtypes após processamento:')
print(X_val_processed.dtypes.value_counts())

## 4. Análise das Features Transformadas

In [ ]:
# Comparar antes/depois
print('ANTES DO PROCESSAMENTO:')
print(X_train.describe())

print('\nDEPOIS DO PROCESSAMENTO:')
print(X_train_processed.describe())

In [ ]:
# Verificar valores ausentes antes/depois
print('VALORES AUSENTES ANTES:')
print(X_train.isnull().sum().sum())

print('\nVALORES AUSENTES DEPOIS:')
print(X_train_processed.isnull().sum().sum())

## 5. Encoding de Target

In [ ]:
# Encode target
y_train_encoded = preprocessor.encode_target(y_train)
y_val_encoded = preprocessor.encode_target(y_val)

print('Target encoding:')
for original, encoded in preprocessor.target_mapping.items():
    print(f'  {original} -> {encoded}')

print(f'\ny_train_encoded shape: {y_train_encoded.shape}')
print(f'y_train_encoded distribution:')
print(pd.Series(y_train_encoded).value_counts().sort_index())

## 6. Feature Importance (Basal)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# Treinar modelo rápido para feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_processed, y_train_encoded)

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train_processed.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
plt.barh(feature_importance['feature'][:20], feature_importance['importance'][:20])
plt.xlabel('Importância')
plt.ylabel('Feature')
plt.title('Top 20 Features - Random Forest Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('\nTop 20 features:')
print(feature_importance.head(20))

## 7. Análise de Correlação Pós-Processamento

In [ ]:
# Correlação de features processadas (apenas numéricas)
numeric_processed = X_train_processed.select_dtypes(include=[np.number])
correlation_matrix = numeric_processed.corr()

# Encontrar correlações altas
high_corr = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_val = correlation_matrix.iloc[i, j]
        if abs(corr_val) > 0.7:  # Threshold alta correlação
            high_corr.append({
                'Feature 1': correlation_matrix.columns[i],
                'Feature 2': correlation_matrix.columns[j],
                'Correlação': corr_val
            })

if high_corr:
    high_corr_df = pd.DataFrame(high_corr).sort_values('Correlação', key=abs, ascending=False)
    print('Features altamente correlacionadas (|r| > 0.7):')
    display(high_corr_df)
else:
    print('Nenhuma alta correlação encontrada.')

## 8. Distribuição de Features por Classe

In [ ]:
# Adicionar target de volta para análise
train_processed_with_target = X_train_processed.copy()
train_processed_with_target['pathogenicity'] = y_train.values

# Comparar features importantes por classe
top_features = feature_importance['feature'].head(4).tolist()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for idx, feature in enumerate(top_features):
    for class_label in train_processed_with_target['pathogenicity'].unique():
        data = train_processed_with_target[
            train_processed_with_target['pathogenicity'] == class_label
        ][feature]
        axes[idx].hist(data, alpha=0.5, label=class_label, bins=30)
    
    axes[idx].set_title(f'Distribuição: {feature}')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frequência')
    axes[idx].legend()

plt.tight_layout()
plt.show()

## 9. Salvando Preprocessor

In [ ]:
# Salvar preprocessor
preprocessor.save('../models/preprocessor.joblib')

print('Preprocessor salvo em ../models/preprocessor.joblib')

## 10. Testando Carregamento

In [ ]:
# Carregar preprocessor salvo
loaded_preprocessor = VariantPreprocessor.load('../models/preprocessor.joblib')

# Transformar dados com preprocessor carregado
X_val_loaded = loaded_preprocessor.transform(X_val)

# Verificar se é igual
print('Dados transformados iguais?', X_val_processed.equals(X_val_loaded))

## 11. Resumo

In [ ]:
print('\n' + '='*80)
print('RESUMO DE FEATURE ENGINEERING')
print('='*80)

print(f'\nShape original: {X_train.shape}')
print(f'Shape processado: {X_train_processed.shape}')
print(f'Features removidas: {X_train.shape[1] - X_train_processed.shape[1]}')

print(f'\nValores ausentes antes: {X_train.isnull().sum().sum()}')
print(f'Valores ausentes depois: {X_train_processed.isnull().sum().sum()}')

print(f'\nTop 5 features mais importantes:')
for idx, row in feature_importance.head(5).iterrows():
    print(f'  {row["feature"]}: {row["importance"]:.4f}')